# Manufatura Peça Mil

Sejam $x_i$, $i\in[3]$ a quantidade produzida na Fábrica $i$, e $y_{ij}$ a quantidade transportada da Fábrica $i$ ao Atacadista $j$, $j\in[5]$. Seja $c_{ij}$ o custo de transporte por unidade da Fábrica $i$ ao Atacadista $j$, $d_j$ a demanda do Atacadista $j$, e $C_i$ a capacidade da Fábrica $i$.

In [ ]:
using JuMP
using GLPK

## Formulação explicita

In [ ]:
me = Model(GLPK.Optimizer);

In [ ]:
@variable(me, x[1:3] >= 0)
@variable(me, y[1:3,1:5] >= 0);

### Função objetivo

In [ ]:
@objective(me, Max, 0.5*x[1]+1.5*x[2]+0.7*x[3]
    -0.05*y[1,1]-0.07*y[1,2]-0.11*y[1,3]-0.15*y[1,4]-0.15*y[1,5]
    -0.08*y[2,1]-0.06*y[2,2]-0.1*y[2,3]-0.12*y[2,4]-0.15*y[2,5]
    -0.1*y[3,1]-0.09*y[3,2]-0.09*y[3,3]-0.1*y[3,4]-0.16*y[3,5])

### Restrições de demanda

In [ ]:
@constraints(me, begin
    y[1,1]+y[2,1]+y[3,1]==2700
    y[1,2]+y[2,2]+y[3,2]==2700
    y[1,3]+y[2,3]+y[3,3]==9000
    y[1,4]+y[2,4]+y[3,4]==4500
    y[1,5]+y[2,5]+y[3,5]==3600
    end)

### Restrições de capacidade

In [ ]:
@constraints(me, begin
  y[1,1]+y[1,2]+y[1,3]+y[1,4]+y[1,5]<=4500
  y[2,1]+y[2,2]+y[2,3]+y[2,4]+y[2,5]<=9000
  y[3,1]+y[3,2]+y[3,3]+y[3,4]+y[3,5]<=11250
    end)

### Produção iguala distribuição

In [ ]:
@constraints(me, begin
    x[1]==y[1,1]+y[1,2]+y[1,3]+y[1,4]+y[1,5]
    x[2]==y[2,1]+y[2,2]+y[2,3]+y[2,4]+y[2,5]
    x[3]==y[3,1]+y[3,2]+y[3,3]+y[3,4]+y[3,5]
    end)

## Formulação compacta

Custos de tranporte $c$, capacidades $C$, demandas menais $d$, custos de produção $p$:

In [ ]:
c = [[0.05 0.07 0.11 0.15 0.15]; [0.08 0.06 0.10 0.12 0.15]; [0.10 0.09 0.09 0.10 0.16]]
C = [4500,9000,11250]
d = [2700,2700,9000,4500,3600]
p = [2.0,1.0,1.8];

In [ ]:
mc = Model(GLPK.Optimizer);

In [ ]:
@variable(mc, x[1:3] >= 0)
@variable(mc, y[1:3,1:5] >= 0);

### Função objetivo

In [ ]:
@objective(mc, Max, sum((2.5-p[i])*x[i] for i=1:3)-sum(c[i,j]*y[i,j] for i=1:3, j=1:5))

### Restrições de demanda, capacidade, e produção iguala distribuição 

In [ ]:
@constraints(mc, begin
    [j=1:5], sum(y[i,j] for i=1:3)==d[j]
    [i=1:3], sum(y[i,j] for j=1:5)<=C[i]
    [i=1:3], x[i] == sum(y[i,j] for j=1:5)
    end)